# RVC → Piper Training Studio — Google Colab

This notebook runs the headless RVC → Piper pipeline on **CPU or GPU**.

It supports:
- **Auto / GPU / CPU compute selection**;
- Google Drive persistence for datasets/checkpoints/final models;
- RVC `.pth` plus optional `.index`;
- pitch `+12` or any other pitch;
- high-pitch audio cleanup before Piper training;
- warm-started Piper training and resume from `last.ckpt`;
- selectable Alba, Amy, any Piper voice key, or a custom ONNX + JSON base voice;
- final `.onnx` + `.onnx.json` export and an inline TTS test.

**GPU is strongly recommended for training speed, but it is no longer required.**


In [ ]:
#@title 0. Storage and compute mode
compute_mode = "Auto (GPU if available, otherwise CPU)" #@param ["Auto (GPU if available, otherwise CPU)", "GPU", "CPU"]
mount_google_drive = True #@param {type:"boolean"}

import os
import torch

if mount_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")

requested_compute = {
    "Auto (GPU if available, otherwise CPU)": "auto",
    "GPU": "gpu",
    "CPU": "cpu",
}[compute_mode]

cuda_available = torch.cuda.is_available()
if requested_compute == "auto":
    resolved_compute = "gpu" if cuda_available else "cpu"
elif requested_compute == "gpu":
    if not cuda_available:
        raise RuntimeError(
            "GPU mode was selected, but this Colab runtime has no CUDA GPU. "
            "Choose Runtime > Change runtime type > GPU, or select CPU/Auto."
        )
    resolved_compute = "gpu"
else:
    resolved_compute = "cpu"

os.environ["RVC_PIPER_COMPUTE"] = resolved_compute

print("Colab base Torch:", torch.__version__)
print("CUDA available:", cuda_available)
print("Requested compute:", requested_compute)
print("Resolved compute:", resolved_compute)
if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
if resolved_compute == "cpu":
    print("WARNING: Piper training on CPU can be much slower than GPU training.")


## 1. Clone/update the Studio repo

This always pulls the latest `main` branch.


In [ ]:
import os, subprocess

REPO = "/content/RVC-to-Piper-Training-APP"
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone",
        "https://github.com/NekoSuneVR/RVC-to-Piper-Training-APP.git",
        REPO,
    ], check=True)

os.chdir(REPO)
print("Repo:", REPO)


## 2. Install the Colab runtime

The setup script creates isolated Python 3.12 RVC/Piper environments under `/content/rvc-piper-runtime`.

- **GPU mode:** installs CUDA PyTorch and verifies a CUDA kernel.
- **CPU mode:** installs CPU-only PyTorch and does not require `nvidia-smi`.
- **Auto:** chooses GPU when one is attached, otherwise CPU.

If you switch CPU ↔ GPU in the same Colab VM, the environments are rebuilt automatically so CPU/CUDA wheels do not get mixed. RMVPE/HuBERT, Piper native extensions, and the default Alba base voice are installed in either mode.


In [ ]:
import os, subprocess

os.environ["RVC_PIPER_COMPUTE"] = resolved_compute
print("Installing Colab runtime for:", resolved_compute)

result = subprocess.run(
    ["bash", "colab/setup_colab.sh", "/content/RVC-to-Piper-Training-APP"],
    text=True,
)
if result.returncode:
    print("\nSetup failed. Last 160 log lines:\n")
    subprocess.run(["tail", "-n", "160", "/content/rvc-piper-runtime/setup.log"])
    raise RuntimeError(f"Colab setup failed with exit code {result.returncode}")

print("Runtime setup complete for:", resolved_compute)


## 3. Build settings

Put your RVC files in Google Drive, for example:

```text
MyDrive/RVC-Piper-Colab/models/my-voice.pth
MyDrive/RVC-Piper-Colab/models/my-voice.index
```

If `mount_google_drive = False`, you can instead use paths under `/content`, but those files are temporary and disappear when the Colab runtime is deleted.


In [ ]:
#@title Build settings
voice_name = "en_GB-rvc-custom-medium" #@param {type:"string"}
rvc_model = "/content/drive/MyDrive/RVC-Piper-Colab/models/my-voice.pth" #@param {type:"string"}
rvc_index = "" #@param {type:"string"}

base_piper_voice = "Alba Medium (UK) - en_GB-alba-medium" #@param ["Alba Medium (UK) - en_GB-alba-medium", "Amy Medium (US) - en_US-amy-medium", "Other Piper voice key", "Custom ONNX + JSON"]
other_piper_voice_key = "en_US-ryan-medium" #@param {type:"string"}
custom_base_model = "/content/drive/MyDrive/RVC-Piper-Colab/base-voices/custom.onnx" #@param {type:"string"}
custom_base_config = "/content/drive/MyDrive/RVC-Piper-Colab/base-voices/custom.onnx.json" #@param {type:"string"}

pitch = 12 #@param {type:"integer"}
index_rate = 0.75 #@param {type:"number"}
protect = 0.33 #@param {type:"number"}
f0_method = "rmvpe" #@param ["rmvpe", "pm"]

prompt_file = "/content/RVC-to-Piper-Training-APP/data/piper_training_prompts.txt" #@param {type:"string"}
prompt_limit = 120 #@param {type:"integer"}

batch_size = 8 #@param {type:"integer"}
max_epochs = 1000 #@param {type:"integer"}
checkpoint_every = 5 #@param {type:"integer"}
num_workers = 2 #@param {type:"integer"}

drive_root = "/content/drive/MyDrive/RVC-Piper-Colab" #@param {type:"string"}
generate_dataset = True #@param {type:"boolean"}
resume_if_possible = True #@param {type:"boolean"}

print("Compute:", resolved_compute)
print("Voice:", voice_name)
print("RVC:", rvc_model)
print("Pitch:", pitch)
print("Base Piper selection:", base_piper_voice)
print("Project root:", f"{drive_root}/{voice_name}")
if resolved_compute == "cpu":
    print("CPU TIP: use a small test first, e.g. prompt_limit=10 and max_epochs=1.")


## 4. Resolve/download the selected base Piper voice

Alba and Amy are presets. `Other Piper voice key` reads the official Piper `voices.json` manifest. `Custom ONNX + JSON` uses paths you provide.


In [ ]:
from pathlib import Path
from urllib.parse import quote
import json, urllib.request

RUNTIME = Path("/content/rvc-piper-runtime")
BASE_DIR = RUNTIME / "base-voice"
BASE_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    "Alba Medium (UK) - en_GB-alba-medium": "en_GB-alba-medium",
    "Amy Medium (US) - en_US-amy-medium": "en_US-amy-medium",
}

def _download_file(url, destination):
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size > 0:
        return destination
    part = destination.with_suffix(destination.suffix + ".part")
    part.unlink(missing_ok=True)
    print("Downloading:", destination.name)
    urllib.request.urlretrieve(url, part)
    part.replace(destination)
    return destination

if base_piper_voice == "Custom ONNX + JSON":
    base_model = Path(custom_base_model)
    base_config = Path(custom_base_config)
    if not base_model.is_file():
        raise FileNotFoundError(f"Custom Piper ONNX not found: {base_model}")
    if not base_config.is_file():
        raise FileNotFoundError(f"Custom Piper JSON not found: {base_config}")
    selected_piper_key = "custom-model"
else:
    selected_piper_key = (
        other_piper_voice_key.strip()
        if base_piper_voice == "Other Piper voice key"
        else PRESETS[base_piper_voice]
    )
    if not selected_piper_key:
        raise ValueError("Enter a Piper voice key.")

    manifest_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json"
    print("Looking up Piper voice:", selected_piper_key)
    with urllib.request.urlopen(manifest_url, timeout=60) as response:
        voices = json.load(response)

    voice = voices.get(selected_piper_key)
    if voice is None:
        matches = [key for key in voices if selected_piper_key.lower() in key.lower()][:20]
        raise KeyError(
            f"Piper voice key not found: {selected_piper_key}. "
            f"Close matches: {', '.join(matches) if matches else 'none'}"
        )

    files = list(voice.get("files", {}).keys())
    onnx_rel = next((p for p in files if p.endswith(".onnx")), None)
    json_rel = next((p for p in files if p.endswith(".onnx.json")), None)
    if not onnx_rel or not json_rel:
        raise RuntimeError(f"Voice manifest is missing ONNX/JSON files for {selected_piper_key}")

    base_model = BASE_DIR / Path(onnx_rel).name
    base_config = BASE_DIR / Path(json_rel).name
    hf_root = "https://huggingface.co/rhasspy/piper-voices/resolve/main/"
    _download_file(hf_root + quote(onnx_rel, safe="/._-") + "?download=true", base_model)
    _download_file(hf_root + quote(json_rel, safe="/._-") + "?download=true", base_config)

print("Selected base Piper voice:", selected_piper_key)
print("Base model:", base_model)
print("Base config:", base_config)


## 5. Build everything

This runs a one-sentence preflight first, then performs base Piper synthesis → RVC conversion → high-pitch cleanup → Piper training on the selected **CPU/GPU** → best-checkpoint ONNX export.

If a persistent `last.ckpt` exists and `resume_if_possible = True`, training resumes from it.


In [ ]:
import subprocess

RUNTIME = "/content/rvc-piper-runtime"
cmd = [
    "python3",
    "/content/RVC-to-Piper-Training-APP/colab/run_build.py",
    "--repo-root", "/content/RVC-to-Piper-Training-APP",
    "--rvc-root", f"{RUNTIME}/rvc",
    "--rvc-python", f"{RUNTIME}/rvc-venv/bin/python",
    "--piper-python", f"{RUNTIME}/piper-venv/bin/python",
    "--drive-root", drive_root,
    "--voice-name", voice_name,
    "--rvc-model", rvc_model,
    "--prompts", prompt_file,
    "--prompt-limit", str(prompt_limit),
    "--pitch", str(pitch),
    "--index-rate", str(index_rate),
    "--protect", str(protect),
    "--f0-method", f0_method,
    "--batch-size", str(batch_size),
    "--max-epochs", str(max_epochs),
    "--checkpoint-every", str(checkpoint_every),
    "--num-workers", str(num_workers),
    "--accelerator", resolved_compute,
    "--base-model", str(base_model),
    "--base-config", str(base_config),
]
if rvc_index.strip():
    cmd += ["--rvc-index", rvc_index.strip()]
elif index_rate > 0:
    print("No RVC .index selected, so index_rate is being changed to 0.")
    cmd[cmd.index("--index-rate") + 1] = "0"
if not generate_dataset:
    cmd.append("--skip-dataset")
if not resume_if_possible:
    cmd.append("--no-resume")

print("Starting guarded Colab build...")
print("Compute:", resolved_compute)
print("Base Piper:", selected_piper_key)
result = subprocess.run(cmd, check=False)
if result.returncode:
    raise RuntimeError(
        f"Colab build failed with exit code {result.returncode}. "
        "The detailed preflight/build error is printed above and saved in /content/rvc-piper-runtime/build.log."
    )


## 6. Check the exported files


In [ ]:
from pathlib import Path

project = Path(drive_root) / voice_name / "piper"
onnx_path = project / f"{voice_name}.onnx"
config_path = project / f"{voice_name}.onnx.json"

print("ONNX:", onnx_path, f"{onnx_path.stat().st_size / 1024**2:.1f} MB" if onnx_path.exists() else "MISSING")
print("JSON:", config_path, config_path.exists())
if not onnx_path.exists() or not config_path.exists():
    raise RuntimeError("The final Piper ONNX/JSON pair was not found yet.")


## 7. Test the finished standalone Piper voice

This is pure Piper inference — no RVC conversion after synthesis.


In [ ]:
#@title Test text
test_text = "Hello! This is my new standalone Piper voice running from Google Colab." #@param {type:"string"}

import subprocess
from IPython.display import Audio, display
from pathlib import Path

test_wav = Path("/content/test-custom-piper.wav")
piper_python = "/content/rvc-piper-runtime/piper-venv/bin/python"
subprocess.run([
    piper_python, "-m", "piper",
    "--model", str(onnx_path),
    "--config", str(config_path),
    "--output-file", str(test_wav),
    "--", test_text,
], check=True)
display(Audio(str(test_wav)))
print("Test WAV:", test_wav)


## Notes

- **Auto** is recommended: it uses a Colab GPU when attached and falls back to CPU otherwise.
- **GPU** explicitly requires CUDA.
- **CPU** installs CPU-only PyTorch and works without an NVIDIA runtime, but Piper training can be dramatically slower.
- Switching CPU ↔ GPU causes the managed RVC/Piper virtual environments to be rebuilt automatically.
- Alba Medium is the default base voice; Amy and custom/other Piper voices are supported.
- Google Drive is recommended for persistence. `/content` can be used for temporary storage if you disable Drive mounting, but it is lost when the runtime is deleted.
- If GPU VRAM is limited, reduce `batch_size`. For CPU testing, start with `prompt_limit=10`, `max_epochs=1`, and `batch_size=2`.
